# 🚀 Imports

In [3]:
import pandas as pd

# String
import urllib
import tldextract
import ipaddress
import re

# Misc
import sys

def print_safe(url):
    '''Prints safe, unclickable URL in VS Code.'''
    sys.stdout.write(url.replace("://", "://\x00"))  # Null byte
    
# Number of CPU cores
# !lscpu

# ❗ IMPORTANT: SAFETY GUIDELINES AGAINST LINKS ⚠️

> ***READ BEFORE RUNNING ANY CELL***
>
> **The `url` column is dangerous if click link**, be careful. Other columns from are safe.
> - ⚠️ Display `df` — Always use `display(df)` so that URL are  not clickable.
> - ⚠️ Use `print_safe` — Use `print_safe(url)` so that URL are  not clickable.
> - ⚠️ String Parsing — Safe but make sure not to accidentally click link
>   - `urlparse`— Splits URL by structure
>   - `tldextract`— Splits domain specifically
>   - `re` — Finds patterns in string
> - ❌ DNS/WHOIS lookups — use `safe_dns_lookup()` only, one person runs this
> - ❌ `requests.get(url)` — never, under any circumstance


In [4]:
url = "http://mail.paypa1-secure.evil.com/login?user=1%20x"

# urlparse
from urllib.parse import urlparse
p = urlparse(url)
p.scheme    # 'http'
p.netloc    # 'mail.paypa1-secure.evil.com'
p.path      # '/login'
p.query     # 'user=1%20x'

# tldextract
ext = tldextract.extract(url)
ext.subdomain  # 'mail.paypa1-secure'
ext.domain     # 'evil'
ext.suffix     # 'com'

# re
re.search(r'\d+\.\d+\.\d+\.\d+', url)  # IP address present?
re.findall(r'\d', url)                 # digits in URL
re.search(r'%[0-9a-fA-F]{2}', url)     # hex encoding present?

print_safe(url)

http:// mail.paypa1-secure.evil.com/login?user=1%20x

---

# 🧪  Import Data

In [ ]:
# Link to CSV
file_path = "../data/dataset-phishing.csv"
df = pd.read_csv(file_path)

# Move all columns with strings to the left
foo = ['url', 'dom', 'tld', 'url_len', 'dom_len', 'tld_len']
cols = foo + [col for col in df.columns if col not in foo]
df = df.reindex(columns=cols)
df.head()

,url,dom,tld,url_len,dom_len,tld_len,is_ip,subdom_cnt,letter_cnt,digit_cnt,...,under_cnt,letter_ratio,digit_ratio,spec_ratio,is_https,slash_cnt,entropy,path_len,query_len,label
0,https://www.rmit.edu.au/,rmit.edu.au,edu.au,24,11,6,0,1,17,0,...,0,0.708333,0.0,0.291667,1,3,3.709148,1,0,0
1,http://www.latrobe.edu.au/,latrobe.edu.au,edu.au,26,14,6,0,1,19,0,...,0,0.730769,0.0,0.269231,0,3,3.738149,1,0,0
2,https://www.cqu.edu.au/,cqu.edu.au,edu.au,23,10,6,0,1,16,0,...,0,0.695652,0.0,0.304348,1,3,3.609668,1,0,0
3,http://bond.edu.au/,bond.edu.au,edu.au,19,11,6,0,0,13,0,...,0,0.684211,0.0,0.315789,0,3,3.576618,1,0,0
4,http://www.csu.edu.au/,csu.edu.au,edu.au,22,10,6,0,1,15,0,...,0,0.681818,0.0,0.318182,0,3,3.503998,1,0,0


---

# 🛠️ Parsing Functions

There is a single URL data that is misconfigured, other than that everything else is consistent.

In [84]:
import math
from collections import Counter
from urllib.parse import urlparse
import ipaddress
import tldextract


def _character_features(url: str, url_len: int) -> dict:
    """
    Internal helper – computes all character‑based features (6‑17, 19‑20)
    in a single pass over the URL string.
    """
    letter_cnt = digit_cnt = special_cnt = 0
    # -- Feature 9: eq_cnt, 10: qm_cnt, 11: amp_cnt,
    # 12: dot_cnt, 13: dash_cnt, 14: under_cnt, 19: slash_cnt --
    eq = qm = amp = dot = dash = under = slash = 0
    freq = Counter()

    for ch in url:
        # -- Feature 6: letter_cnt --
        if ch.isalpha(): letter_cnt += 1
        # -- Feature 7: digit_cnt --
        elif ch.isdigit(): digit_cnt += 1
        # -- Feature 8: special_cnt --
        else: special_cnt += 1

        # -- Feature 9: eq_cnt --
        if ch == '=': eq += 1
        # -- Feature 10: qm_cnt --
        elif ch == '?': qm += 1
        # -- Feature 11: amp_cnt --
        elif ch == '&': amp += 1
        # -- Feature 12: dot_cnt --
        elif ch == '.': dot += 1
        # -- Feature 13: dash_cnt --
        elif ch == '-': dash += 1
        # -- Feature 14: under_cnt --
        elif ch == '_': under += 1
        # -- Feature 19: slash_cnt --
        elif ch == '/': slash += 1

        # -- For Feature 20: entropy (histogram) --
        freq[ch] += 1

    # -- Feature 15: letter_ratio --
    letter_ratio = letter_cnt / url_len if url_len else 0.0 
    # -- Feature 16: digit_ratio --
    digit_ratio  = digit_cnt / url_len if url_len else 0.0
    # -- Feature 17: spec_ratio --
    spec_ratio   = special_cnt / url_len if url_len else 0.0

    # -- Feature 20: entropy --
    if url_len == 0:
        entropy = 0.0
    else:
        entropy = 0.0
        for count in freq.values():
            p = count / url_len
            entropy -= p * math.log2(p)

    return {
        'letter_cnt': letter_cnt,     # Feature 6
        'digit_cnt': digit_cnt,       # Feature 7
        'special_cnt': special_cnt,   # Feature 8
        'eq_cnt': eq,                 # Feature 9
        'qm_cnt': qm,                 # Feature 10
        'amp_cnt': amp,               # Feature 11
        'dot_cnt': dot,               # Feature 12
        'dash_cnt': dash,             # Feature 13
        'under_cnt': under,           # Feature 14
        'letter_ratio': letter_ratio, # Feature 15
        'digit_ratio': digit_ratio,   # Feature 16
        'spec_ratio': spec_ratio,     # Feature 17
        'slash_cnt': slash,           # Feature 19
        'entropy': entropy            # Feature 20
    }


def extract_features(url: str) -> dict:
    """
    Main public function – returns 22 URL features (labelled 1‑22).
    """
    url = str(url)
    parsed = urlparse(url)
    ext = tldextract.extract(url)

    # -- Ignored features --
    dom = ext.top_domain_under_public_suffix
    tld = ext.suffix

    # -- Feature 1: url_len  --
    url_len = len(url)

    # -- Feature 4: is_ip --
    is_ip = 0
    if dom == '':
        try:
            if parsed.hostname:
                dom = str(ipaddress.ip_address(parsed.hostname))
                is_ip = 1
        except ValueError:
            dom = ext.suffix

    # -- Feature 2: dom_len --
    dom_len = len(dom)
    # -- Feature 3: tld_len --
    tld_len = len(tld)

    # -- Feature 5: subdom_cnt --
    sub = ext.subdomain
    subdom_cnt = len(sub.split('.')) if sub else 0

    # -- Features 6‑17, 19‑20 --
    char_feats = _character_features(url, url_len)

    # -- Feature 18: is_https --
    is_https = 1 if parsed.scheme == 'https' else 0

    # -- Feature 21: path_len --
    path_len = len(parsed.path)
    # -- Feature 22: query_len --
    query_len = len(parsed.query)

    # Return all 22 features in order 1‑22
    return {
        'url_len': url_len,           # Feature 1
        'dom_len': dom_len,           # Feature 2
        'tld_len': tld_len,           # Feature 3
        'is_ip': is_ip,               # Feature 4
        'subdom_cnt': subdom_cnt,     # Feature 5
        **char_feats,                 # Features 6‑17, 19‑20
        'is_https': is_https,         # Feature 18
        'path_len': path_len,         # Feature 21
        'query_len': query_len        # Feature 22
    }

In [85]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Apply the function to every URL in the DataFrame
# ------------------------------------------------------------
computed_series = df['url'].apply(extract_features)        # Series of dicts
computed = pd.DataFrame(computed_series.tolist(), index=df.index)  # features as a DataFrame

# ------------------------------------------------------------
# 2. Compare each feature with the corresponding df column
#    (only for columns that exist in both)
# ------------------------------------------------------------
mismatch_summary = {}

for col in computed.columns:
    # Skip if the column doesn't exist in the original DataFrame
    if col not in df.columns:
        print(f"⚠️  Column '{col}' not found in df – skipped.")
        continue

    # Get the two series to compare (same index)
    orig_vals = df[col]
    comp_vals = computed[col]

    # Decide comparison method based on dtype
    if orig_vals.dtype == np.float64 or orig_vals.dtype == np.float32:
        # Float comparison with tolerance (avoid rounding mismatches)
        # Note: isclose can't handle None/NaN well, so fill those gently
        mask = np.isclose(orig_vals.fillna(0), comp_vals.fillna(0), atol=1e-12)
    else:
        # Integer or other – simple equality
        mask = (orig_vals == comp_vals)

    # Count mismatches
    n_mismatch = (~mask).sum()
    mismatch_summary[col] = n_mismatch

    # If you'd like to see the mismatched rows for debugging, uncomment:
    # if n_mismatch > 0:
    #     print(f"\nMismatches in '{col}':")
    #     bad = df.loc[~mask, ['url', col]].copy()
    #     bad['computed'] = comp_vals[~mask]
    #     print(bad.to_string())

# ------------------------------------------------------------
# 3. Print summary
# ------------------------------------------------------------
print("Mismatch count per feature:")
for col, cnt in mismatch_summary.items():
    print(f"  {col:20s} : {cnt:6d} mismatches")

total_mismatches = sum(mismatch_summary.values())
print(f"\nTotal mismatched rows across all features: {total_mismatches}")

Mismatch count per feature:
  url_len              :      0 mismatches
  dom_len              :      1 mismatches
  tld_len              :      0 mismatches
  is_ip                :      0 mismatches
  subdom_cnt           :      0 mismatches
  letter_cnt           :      0 mismatches
  digit_cnt            :      0 mismatches
  special_cnt          :      0 mismatches
  eq_cnt               :      0 mismatches
  qm_cnt               :      0 mismatches
  amp_cnt              :      0 mismatches
  dot_cnt              :      0 mismatches
  dash_cnt             :      0 mismatches
  under_cnt            :      0 mismatches
  letter_ratio         :      0 mismatches
  digit_ratio          :      0 mismatches
  spec_ratio           :      0 mismatches
  slash_cnt            :      0 mismatches
  entropy              :      0 mismatches
  is_https             :      0 mismatches
  path_len             :      0 mismatches
  query_len            :      0 mismatches

Total mismatched rows acr

---

# Testing

In [71]:
import math
from collections import Counter

def shannon_entropy(url: str) -> float:
    """Return the Shannon entropy (in bits) of the URL string."""
    s = str(url)
    if not s:
        return 0.0
    freq = Counter(s) # dict-like: char -> count
    length = len(s)
    entropy = 0.0
    for count in freq.values():
        p = count / length
        entropy -= p * math.log2(p)
    return entropy

In [86]:

# url = 'https://www.rmit.edu.au/'
# url = 'http://202.194.232.100:8005/english/index.asp'
# url = 'http://gov.ug/'
# url = 'http://www.ippt.pan.pl/en/'
# url = 'https://www.sztaki.hu/?en'
url = 'https://гуманитарный-институт.рф'

p = urlparse(url)
ext = tldextract.extract(url)

# Feature 1 (url)
url_len = len(url) 

# Feature 2 (dom)
dom =  ext.top_domain_under_public_suffix # Extract domain
# Feature 7 (is_ip)
is_ip = 0
# If domain is empty
if dom == '':
    try: # If IP address, add as domain
        dom = str(ipaddress.ip_address(p.hostname))
        is_ip = 1
    
    # If no domain, add suffix as domain
    except ValueError: 
        dom = ext.suffix

# Feature 3 (tld)
tld = ext.suffix

# Feature 4 (url_len), Feature 5 (dom_len), Feature 6 (tld_len)
url_len = len(url)
dom_len = len(dom)
tld_len = len(tld)

# Feature 8  (subdom_cnt)
sub = ext.subdomain
subdom_cnt = len(sub.split('.')) if sub else 0

# Feature 9 (letter_cnt), Feature 10 (digit_cnt), Feature 11 (special_cnt)
letter_cnt = sum(1 for c in url if c.isalpha())          # letters (Unicode)
digit_cnt  = sum(1 for c in url if c.isdigit())          # digits (Unicode)
special_cnt = sum(1 for c in url if not c.isalpha() and not c.isdigit())  # non-alphanumeric


# Feature 12 (eq_cnt), Feature 13 (qm_cnt), Feature 14 (amp_cnt), Feature 15 (dot_cnt), Feature 16 (dash_cnt), Feature 17 (under_cnt)
eq_cnt = url.count('=')
qm_cnt = url.count('?')
amp_cnt = url.count('&')
dot_cnt = url.count('.')
dash_cnt = url.count('-')
under_cnt = url.count('_')

# Feature 18 (letter_ratio), Feature 19 (digit_ratio), Feature 20 (spec_ratio)
letter_ratio = letter_cnt / url_len if url_len > 0 else 0
digit_ratio = digit_cnt / url_len if url_len > 0 else 0
spec_ratio = special_cnt / url_len if url_len > 0 else 0

# Feature 21 (is_https)
is_https = 1 if p.scheme == 'https' else 0

# Feature 22 (slash_cnt)
slash_cnt = url.count('/')

# Feature 23 (entropy)
entropy = shannon_entropy(url)

# Feature 24 (path_len)
path_len = len(p.path)

# Feature 25 (query_len)
query_len = len(p.query)

print(entropy)

4.140319531114783


In [87]:
import numpy as np

# df[df['dom'] == 'rmit.edu.au'].iloc[:, np.r_[0, 22:26]]
# df[df['dom'] == '202.194.232.100'].iloc[:, np.r_[0, 22:26]]
# df[df['dom'] == 'gov.ug'].iloc[:, np.r_[0, 22:26]]
# df[df['url'] == 'http://www.ippt.pan.pl/en/'].iloc[:, np.r_[0, 22:26]]
# df[df['url'] == 'https://www.sztaki.hu/?en'].iloc[:, np.r_[0, 22:26]]
df[df['url'] == 'https://гуманитарный-институт.рф'].iloc[:, np.r_[0, 22:26]]



,url,entropy,path_len,query_len,label
90889,https://гуманитарный-институт.рф,4.14032,0,0,0


---